In [1]:
# Install uv package manager
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 108.5 MB/s eta 0:00:0000:0100:01


In [ ]:
!uv pip install langchain_docling docling langchain_openai langchain_qdrant langchain_text_splitters

Using Python 3.12.12 environment at: /usr
Resolved 123 packages in 1.30s                                       
Prepared 28 packages in 769ms                                            
Uninstalled 1 package in 2ms
Installed 28 packages in 63ms                               
 + colorlog==6.10.1
 + docling==2.68.0
 + docling-core==2.59.0
 + docling-ibm-models==3.10.3
 + docling-parse==4.7.3
 + faker==40.1.2
 + filetype==1.2.0
 + jsonlines==4.0.0
 + jsonref==1.1.0
 + langchain-docling==2.0.0
 + latex2mathml==3.78.1
 + marko==2.2.2
 + mpire==2.10.2
 + polyfactory==3.2.0
 + pyclipper==1.4.0
 + pylatexenc==2.10
 + pypdfium2==4.30.0
 + python-docx==1.2.0
 + python-pptx==1.0.2
 + rapidocr==3.5.0
 + semchunk==2.2.2
 + tree-sitter==0.25.2
 + tree-sitter-c==0.24.1
 + tree-sitter-javascript==0.25.0
 + tree-sitter-python==0.25.0
 + tree-sitter-typescript==0.23.2
 - typer==0.21.1
 + typer==0.19.2
 + xlsxwriter==3.2.9


## Docling Quick Testing

In [15]:
from docling.document_converter import DocumentConverter

source = "https://arxiv.org/pdf/2408.09869"  # file path or URL
converter = DocumentConverter()
doc = converter.convert(source).document

print(doc.export_to_markdown())  # output: "### Docling Technical Report[...]"

[INFO] 2026-01-15 15:57:18,804 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-15 15:57:18,805 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-15 15:57:18,846 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-15 15:57:18,847 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-15 15:57:19,060 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-15 15:57:19,061 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-15 15:57:19,065 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-01-15 15:57:19,066 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-01-1

<!-- image -->

## Docling Technical Report

## Version 1.0

Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar

AI4K Group, IBM Research R¨ uschlikon, Switzerland

## Abstract

This technical report introduces Docling , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility and addition of new features and models.

## 1 Introduction

Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge vari

In [ ]:
# Check current directory in colab
import os
print(os.getcwd())  # Shows: /content
print(os.path.abspath("docling_technical_report.md"))  # Shows: /content/docling_technical_report.md

/content
/content/docling_technical_report.md


In [ ]:
# Save the document to a file (available in colab folder)
with open("docling_technical_report.md", "w") as f:
    f.write(doc.export_to_markdown())

# Download to local machine (directly download from colab)
from google.colab import files
files.download("docling_technical_report.md")

Update (16/01): 

This is already confirmed in the colab that files is saved in colab folder.

## Docling Paper Ingestion

In [ ]:
import os
from langchain_docling.loader import DoclingLoader, ExportType
from docling.chunking import HybridChunker
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

# https://github.com/huggingface/transformers/issues/5486:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OPENAI_API_KEY"] = ""

# Configuration
FILE_PATH = ["https://arxiv.org/pdf/2408.09869"]  # Docling Technical Report
EMBED_MODEL = "text-embedding-3-small"  # OpenAI embedding model
EXPORT_TYPE = ExportType.DOC_CHUNKS
QDRANT_PATH = "./qdrant_storage"  # Local persistent storage
COLLECTION_NAME = "docling_demo"

print("Loading documents with Docling...")
loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=EXPORT_TYPE,
    chunker=HybridChunker(tokenizer="sentence-transformers/all-MiniLM-L6-v2"),
)

docs = loader.load()
print(f"Loaded {len(docs)} document chunks")

# Determine splits
if EXPORT_TYPE == ExportType.DOC_CHUNKS:
    splits = docs
elif EXPORT_TYPE == ExportType.MARKDOWN:
    from langchain_text_splitters import MarkdownHeaderTextSplitter

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header_1"),
            ("##", "Header_2"),
            ("###", "Header_3"),
        ],
    )
    splits = [split for doc in docs for split in splitter.split_text(doc.page_content)]
else:
    raise ValueError(f"Unexpected export type: {EXPORT_TYPE}")

# Sample splits
print("\nSample splits:")
for d in splits[:3]:
    print(f"- {d.page_content[:100]}...")
print("...")

# Initialize OpenAI embeddings
print("\nInitializing OpenAI embeddings...")
embedding = OpenAIEmbeddings(model=EMBED_MODEL)

# Create Qdrant vectorstore with local storage
print("Creating Qdrant vectorstore...")
vectorstore = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embedding,
    path=QDRANT_PATH,
    collection_name=COLLECTION_NAME,
)

print(f"\n✅ Successfully created vectorstore with {len(splits)} chunks!")
print(f"📁 Database saved to: {QDRANT_PATH}")


Update (16/01): 

This is already confirmed in the colab that it will create the qrant index folder (qdrant_storage) in colab folder.

In [ ]:
docs

[Document(metadata={'source': 'https://arxiv.org/pdf/2408.09869', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 113.643, 't': 481.532, 'r': 498.359, 'b': 439.849, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 295]}]}, {'self_ref': '#/texts/4', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 249.283, 't': 427.545, 'r': 362.717, 'b': 408.084, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 50]}]}], 'headings': ['Version 1.0'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 11465328351749295394, 'filename': '2408.09869v5.pdf'}}}, page_content='Version 1.0\nChristoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbaue

In [ ]:
data = docs[2].model_dump()
display(data)

page_content = data['page_content']
print(page_content)

{'id': None,
 'metadata': {'source': 'https://arxiv.org/pdf/2408.09869',
  'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta',
   'version': '1.0.0',
   'doc_items': [{'self_ref': '#/texts/8',
     'parent': {'$ref': '#/body'},
     'children': [],
     'content_layer': 'body',
     'label': 'text',
     'prov': [{'page_no': 1,
       'bbox': {'l': 108.0,
        't': 239.37,
        'r': 504.003,
        'b': 143.54600000000005,
        'coord_origin': 'BOTTOMLEFT'},
       'charspan': [0, 792]}]},
    {'self_ref': '#/texts/9',
     'parent': {'$ref': '#/body'},
     'children': [],
     'content_layer': 'body',
     'label': 'text',
     'prov': [{'page_no': 1,
       'bbox': {'l': 108.0,
        't': 135.88800000000003,
        'r': 504.003,
        'b': 83.52099999999996,
        'coord_origin': 'BOTTOMLEFT'},
       'charspan': [0, 488]}]},
    {'self_ref': '#/texts/12',
     'parent': {'$ref': '#/body'},
     'children': [],
     'content_layer': 'body',
     'l

1 Introduction
Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge variability in formats, weak standardization and printing-optimized characteristic, which discards most structural features and metadata. With the advent of LLMs and popular application patterns such as retrieval-augmented generation (RAG), leveraging the rich content embedded in PDFs has become ever more relevant. In the past decade, several powerful document understanding solutions have emerged on the market, most of which are commercial software, cloud offerings [3] and most recently, multi-modal vision-language models. As of today, only a handful of open-source tools cover PDF conversion, leaving a significant feature and quality gap to proprietary solutions.
With Docling , we open-source a very capable and efficient document conversion tool which builds on the powerful, specialized AI models and datasets for layout analysis and table structure rec